# Importing libraries

In [2]:
import wandb
import pandas as pd
import ydata_profiling as pandas_profiling # pandas-profiling error: https://stackoverflow.com/questions/76667611/pydanticimporterror-basesettings-has-been-moved-to-the-pydantic-settings-pa

# Download and read data

In [4]:
run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)

wandb: Currently logged in as: tobias-kuse. Use `wandb login --relogin` to force relogin


In [5]:
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,9138664,Private Lg Room 15 min to Manhattan,47594947,Iris,Queens,Sunnyside,40.74271,-73.92493,Private room,74,2,6,2019-05-26,0.13,1,5
1,31444015,TIME SQUARE CHARMING ONE BED IN HELL'S KITCHEN...,8523790,Johlex,Manhattan,Hell's Kitchen,40.76682,-73.98878,Entire home/apt,170,3,0,NaN,NaN,1,188
2,8741020,Voted #1 Location Quintessential 1BR W Village...,45854238,John,Manhattan,West Village,40.73631,-74.00611,Entire home/apt,245,3,51,2018-09-19,1.12,1,0
3,34602077,Spacious 1 bedroom apartment 15min from Manhattan,261055465,Regan,Queens,Astoria,40.76424,-73.92351,Entire home/apt,125,3,1,2019-05-24,0.65,1,13
4,23203149,Big beautiful bedroom in huge Bushwick apartment,143460,Megan,Brooklyn,Bushwick,40.69839,-73.92044,Private room,65,2,8,2019-06-23,0.52,2,8


# Perform automatic EDA

In [6]:
profile = pandas_profiling.ProfileReport(df)
profile.to_widgets()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

/home/work/miniconda3/envs/mlflow-568ec1b8721af261e41046eb5823b93c05b83b6d/lib/python3.10/site-packages/seaborn/matrix.py:256: FutureWarning: Format strings passed to MaskedConstant are ignored, but in future may error or produce different behavior
  annotation = ("{:" + self.fmt + "}").format(val)
/home/work/miniconda3/envs/mlflow-568ec1b8721af261e41046eb5823b93c05b83b6d/lib/python3.10/site-packages/ydata_profiling/model/missing.py:78: UserWarning: There was an attempt to generate the Heatmap missing values diagrams, but this failed.
To hide this warning, disable the calculation
(using `df.profile_report(missing_diagrams={"Heatmap": False}`)
If this is problematic for your use case, please report this as an issue:
https://github.com/ydataai/ydata-profiling/issues
(include the error message: 'could not convert string to float: '--'')
  warnings.warn(


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render widgets:   0%|          | 0/1 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



,availability_365,calculated_host_listings_count,host_id,id,latitude,longitude,minimum_nights,neighbourhood_group,number_of_reviews,price,reviews_per_month,room_type
availability_365,1.000000,0.407154,0.172829,0.164731,-0.009651,0.061113,0.081264,0.075558,0.237886,0.091799,0.387509,0.088758
calculated_host_listings_count,0.407154,1.000000,0.144830,0.133321,0.002377,0.062861,0.063765,0.086084,0.057464,-0.101398,0.144355,0.095414
host_id,0.172829,0.144830,1.000000,0.558145,0.041903,0.106627,-0.138851,0.098353,-0.117507,-0.073509,0.275662,0.092035
id,0.164731,0.133321,0.558145,1.000000,-0.001193,0.066518,-0.068274,0.059137,-0.302146,-0.026606,0.368858,0.070785
latitude,-0.009651,0.002377,0.041903,-0.001193,1.000000,0.040318,0.017124,0.538339,-0.038288,0.138701,-0.027334,0.111624
longitude,0.061113,0.062861,0.106627,0.066518,0.040318,1.000000,-0.121498,0.648797,0.080778,-0.436764,0.117056,0.153825
minimum_nights,0.081264,0.063765,-0.138851,-0.068274,0.017124,-0.121498,1.000000,0.000000,-0.159285,0.103245,-0.288277,0.020005
neighbourhood_group,0.075558,0.086084,0.098353,0.059137,0.538339,0.648797,0.000000,1.000000,-0.013559,0.119665,0.049929,0.124914
number_of_reviews,0.237886,0.057464,-0.117507,-0.302146,-0.038288,0.080778,-0.159285,-0.013559,1.000000,-0.050723,0.701699,0.000000
price,0.091799,-0.101398,-0.073509,-0.026606,0.138701,-0.436764,0.103245,0.119665,-0.050723,1.000000,-0.018707,0.022859


# Fix some problems with the data

In [7]:
# Drop outliers
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()
# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

# Checking the results

In [8]:
df.describe()

# The minimum and maximum values for price have changed as expected

,id,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
count,1.900100e+04,1.900100e+04,19001.000000,19001.000000,19001.000000,19001.000000,19001.000000,15243.000000,19001.000000,19001.000000
mean,1.883041e+07,6.639459e+07,40.728063,-73.950827,122.340456,6.906900,23.797747,1.380928,6.583811,109.725383
std,1.096986e+07,7.782663e+07,0.055389,0.046825,71.530346,21.456544,45.493455,1.689988,31.154750,130.599899
min,2.539000e+03,2.571000e+03,40.508730,-74.239140,10.000000,1.000000,0.000000,0.010000,1.000000,0.000000
25%,9.355498e+06,7.728754e+06,40.688820,-73.982050,66.000000,1.000000,1.000000,0.190000,1.000000,0.000000
50%,1.938754e+07,3.048785e+07,40.721710,-73.954630,100.000000,2.000000,6.000000,0.720000,1.000000,39.000000
75%,2.891952e+07,1.048354e+08,40.763210,-73.934490,160.000000,5.000000,24.000000,2.010000,2.000000,219.000000
max,3.648561e+07,2.742733e+08,40.913060,-73.717950,350.000000,1250.000000,607.000000,27.950000,327.000000,365.000000


In [9]:
df.info()

# The data type for last_review has been fixed

<class 'pandas.core.frame.DataFrame'>
Int64Index: 19001 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              19001 non-null  int64         
 1   name                            18994 non-null  object        
 2   host_id                         19001 non-null  int64         
 3   host_name                       18993 non-null  object        
 4   neighbourhood_group             19001 non-null  object        
 5   neighbourhood                   19001 non-null  object        
 6   latitude                        19001 non-null  float64       
 7   longitude                       19001 non-null  float64       
 8   room_type                       19001 non-null  object        
 9   price                           19001 non-null  int64         
 10  minimum_nights                  19001 non-null  int64         
 11  nu

In [10]:
run.finish()